In [1]:
import pandas as pd
import random
import os
from pyecharts import options as opts
from pyecharts.charts import Graph

# ================= ⚡ 配置区 =================
TRIPLETS_FILE = 'adni_knowledge_triplets.csv'
OUTPUT_FULL = "adni_kg_full_v2.html"       
OUTPUT_SUBSET = "adni_kg_subset_10_v2.html" 

# 🎨 颜色配置 (修改版：区分 PrimeKG 内部的基因和病)
COLORS = {
    "Patient":         "#c23531",   # 红 (病人核心)
    "Gene":            "#52c41a",   # 🟢 鲜绿 (ADNI 提取的风险基因，如 APOE_e4)
    "Symptom":         "#2f4554",   # 深蓝 (NPI-Q 症状)
    "Level":           "#722ed1",   # 紫色 (严重等级)
    
    # === 修改部分：PrimeKG 拆分为两色 ===
    "PrimeKG_Disease": "#e6a23c",   # 🟠 深橙色 (疾病/表型/通路 - 概念类)
    "PrimeKG_Gene":    "#fdd835",   # 🟡 浅黄色/亮黄 (PrimeKG里的基因 - 实体类)
    # ==================================
    
    "Attribute":       "#61a0a8",   # 青色 (评分MMSE/CDR + 年龄/性别)
    "Other":           "#d48265"    # 其他
}

# ================= 1. 数据加载与预处理 =================
print(f"🚀 正在加载 ADNI 图谱数据: {TRIPLETS_FILE} ...")
if not os.path.exists(TRIPLETS_FILE):
    print("❌ 错误：找不到文件，请确保 CSV 文件在当前目录下")
    exit()

df = pd.read_csv(TRIPLETS_FILE)
df['head'] = df['head'].astype(str)
df['tail'] = df['tail'].astype(str)
df['relation'] = df['relation'].astype(str)

print(f"📊 数据加载完成，共 {len(df)} 条边")


# ================= 2. 辅助函数：确定节点类别 (逻辑升级) =================
def get_node_category(node_name):
    if node_name.startswith("Patient:"): return "Patient"
    if node_name.startswith("Gene:"):    return "Gene"      # ADNI 本地基因
    if node_name.startswith("Symptom:"): return "Symptom"   
    if node_name.startswith("Level:"):   return "Level"     
    
    # === 修改部分：智能区分 PrimeKG 内部类型 ===
    if node_name.startswith("PrimeKG:"): 
        # 提取冒号后面的名字
        real_name = node_name.split(":", 1)[1]
        
        # 🧪 启发式规则：
        # 1. 如果包含空格 (如 'Alzheimer disease') -> 肯定是 Disease/Phenotype
        # 2. 如果是纯大写且没有空格 (如 'DHX9', 'STAU1') -> 认为是 Gene
        # 3. 其他情况归为 Disease
        if " " not in real_name and real_name.isupper() and len(real_name) < 10:
            return "PrimeKG_Gene"     # 🟡 基因
        else:
            return "PrimeKG_Disease"  # 🟠 疾病/其他
    # ==========================================

    if node_name.startswith("Concept:") or node_name.startswith("Test:"): 
        return "Attribute"
    return "Other"

# ================= 3. 核心绘图函数 =================
def render_graph(dataframe, filename, title):
    print(f"\n🎨 正在渲染图谱: {title} -> {filename} ...")
    
    nodes_set = set(dataframe['head']) | set(dataframe['tail'])
    triplets = dataframe.values.tolist()

    # 统计打印
    stats = {}
    for node in nodes_set:
        cat = get_node_category(node)
        stats[cat] = stats.get(cat, 0) + 1
    
    print("-" * 40)
    print(f"📊 图谱成分统计 (总节点: {len(nodes_set)})")
    for cat, count in sorted(stats.items(), key=lambda x: x[1], reverse=True):
        icon = "⚪"
        if "Gene" in cat: icon = "🧬"
        elif "Disease" in cat: icon = "💊"
        elif "Patient" in cat: icon = "👤"
        print(f"   - {icon} {cat:<15}: {count:>5} 个")
    print("-" * 40)
    
    # 2. 构建 ECharts 节点
    echarts_nodes = []
    
    # === 修改部分：更新图例配置 ===
    categories_def = [
        {"name": "Patient",         "itemStyle": {"color": COLORS["Patient"]}},
        {"name": "Gene",            "itemStyle": {"color": COLORS["Gene"]}},
        {"name": "PrimeKG_Disease", "itemStyle": {"color": COLORS["PrimeKG_Disease"]}}, # 新增
        {"name": "PrimeKG_Gene",    "itemStyle": {"color": COLORS["PrimeKG_Gene"]}},    # 新增
        {"name": "Symptom",         "itemStyle": {"color": COLORS["Symptom"]}},
        {"name": "Attribute",       "itemStyle": {"color": COLORS["Attribute"]}},
        {"name": "Level",           "itemStyle": {"color": COLORS["Level"]}},
        {"name": "Other",           "itemStyle": {"color": COLORS["Other"]}},
    ]
    
    cat_map = {c["name"]: i for i, c in enumerate(categories_def)}

    for node in nodes_set:
        cat_name = get_node_category(node)
        cat_idx = cat_map.get(cat_name, 7) # 默认为 Other
        
        # 节点大小微调
        symbol_size = 10
        if cat_name == "Patient": symbol_size = 25
        elif cat_name == "Gene": symbol_size = 22             # ADNI 核心基因大一点
        elif cat_name == "PrimeKG_Disease": symbol_size = 18  # 疾病节点中等
        elif cat_name == "PrimeKG_Gene": symbol_size = 14     # 外部基因稍微小一点，做背景
        elif cat_name == "Symptom": symbol_size = 12
        
        echarts_nodes.append({
            "name": node,
            "symbolSize": symbol_size,
            "category": cat_idx,
            "draggable": True,
            "label": {"show": False},
            "tooltip": {"formatter": "{b}"}
        })

    # 3. 构建连线
    echarts_links = []
    for h, r, t in triplets:
        echarts_links.append({
            "source": h,
            "target": t,
            "value": r,
            "lineStyle": {"width": 0.5, "curveness": 0.1, "opacity": 0.6}
        })

    # 4. 生成
    c = (
        Graph(init_opts=opts.InitOpts(width="100%", height="950px", page_title=title))
        .add(
            "",
            echarts_nodes,
            echarts_links,
            categories=categories_def,
            layout="force",
            repulsion=800,
            gravity=0.1,
            edge_symbol=[None, None],
            edge_label=opts.LabelOpts(is_show=False),
            is_focusnode=True, 
            is_roam=True,
            tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{b} <br/> {c}")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title=title, subtitle=f"节点: {len(nodes_set)} | 边: {len(triplets)}"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_left="2%", pos_top="5%")
        )
    )
    
    c.render(filename)
    print(f"✅ 保存成功: {filename}")

# ================= 4. 任务函数 (保持不变) =================
def generate_subset():
    print("\n" + "=" * 50)
    print("🔍 抽取 10 人子图...")
    all_patients = df[df['head'].str.startswith('Patient:')]['head'].unique()
    if len(all_patients) == 0: return
    selected_patients = random.sample(list(all_patients), min(10, len(all_patients)))
    
    layer1 = df[df['head'].isin(selected_patients)]
    intermediate_nodes = layer1['tail'].unique()
    layer2 = df[df['head'].isin(intermediate_nodes)]
    prime_nodes = layer2[layer2['tail'].str.contains("PrimeKG")]['tail'].unique()
    layer3 = df[(df['head'].isin(prime_nodes)) & (df['tail'].isin(prime_nodes))]
    
    subset_df = pd.concat([layer1, layer2, layer3]).drop_duplicates()
    render_graph(subset_df, OUTPUT_SUBSET, "ADNI 图谱 (PrimeKG 分色版)")

def generate_full():
    print("\n" + "=" * 50)
    print("🔍 准备完整图谱...")
    render_graph(df, OUTPUT_FULL, "ADNI 完整知识图谱 (PrimeKG 分色版)")

if __name__ == "__main__":
    generate_subset()
    generate_full()
    print("\n🎉 完成！")

🚀 正在加载 ADNI 图谱数据: adni_knowledge_triplets.csv ...
📊 数据加载完成，共 36901 条边

🔍 抽取 10 人子图...

🎨 正在渲染图谱: ADNI 图谱 (PrimeKG 分色版) -> adni_kg_subset_10_v2.html ...
----------------------------------------
📊 图谱成分统计 (总节点: 173)
   - ⚪ Other          :   117 个
   - 💊 PrimeKG_Disease:    19 个
   - ⚪ Attribute      :    15 个
   - 👤 Patient        :    10 个
   - ⚪ Symptom        :     8 个
   - ⚪ Level          :     3 个
   - 🧬 Gene           :     1 个
----------------------------------------
✅ 保存成功: adni_kg_subset_10_v2.html

🔍 准备完整图谱...

🎨 正在渲染图谱: ADNI 完整知识图谱 (PrimeKG 分色版) -> adni_kg_full_v2.html ...
----------------------------------------
📊 图谱成分统计 (总节点: 3690)
   - 💊 PrimeKG_Disease:  1662 个
   - 👤 Patient        :   837 个
   - 🧬 PrimeKG_Gene   :   797 个
   - ⚪ Other          :   348 个
   - ⚪ Attribute      :    32 个
   - ⚪ Symptom        :    10 个
   - ⚪ Level          :     3 个
   - 🧬 Gene           :     1 个
----------------------------------------
✅ 保存成功: adni_kg_full_v2.html

🎉 完成！


In [6]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import os

# ================= ⚡ 配置区 =================
TRIPLETS_FILE = 'adni_knowledge_triplets.csv'
OUTPUT_PNG = "adni_kg_subset_high_res.png"
DPI = 600  # 分辨率：600 非常清晰

# 🎨 颜色配置 (保持您的原版配色)
COLORS_MAP = {
    "Patient":         "#c23531",   # 红
    "Gene":            "#52c41a",   # 绿
    "Symptom":         "#2f4554",   # 深蓝
    "Level":           "#722ed1",   # 紫
    "PrimeKG_Disease": "#e6a23c",   # 橙
    "PrimeKG_Gene":    "#fdd835",   # 黄
    "Attribute":       "#61a0a8",   # 青
    "Other":           "#d48265"    # 褐
}

# ================= 1. 辅助函数 (移植版) =================
def get_node_category(node_name):
    node_name = str(node_name)
    if node_name.startswith("Patient:"): return "Patient"
    if node_name.startswith("Gene:"):    return "Gene"
    if node_name.startswith("Symptom:"): return "Symptom"
    if node_name.startswith("Level:"):   return "Level"
    
    if node_name.startswith("PrimeKG:"): 
        try:
            real_name = node_name.split(":", 1)[1]
            if " " not in real_name and real_name.isupper() and len(real_name) < 10:
                return "PrimeKG_Gene"
            else:
                return "PrimeKG_Disease"
        except:
            return "PrimeKG_Disease"

    if node_name.startswith("Concept:") or node_name.startswith("Test:"): 
        return "Attribute"
    return "Other"

# ================= 2. 数据处理与构图 =================
print(f"🚀 正在加载数据: {TRIPLETS_FILE} ...")
if not os.path.exists(TRIPLETS_FILE):
    print("❌ 错误：找不到 CSV 文件")
    exit()

df = pd.read_csv(TRIPLETS_FILE)

# --- 🔍 执行 10 人子图抽取逻辑 (防止节点过多图炸了) ---
print("🔍 正在抽取 10 人子图数据...")
all_patients = df[df['head'].astype(str).str.startswith('Patient:')]['head'].unique()
if len(all_patients) > 0:
    selected_patients = random.sample(list(all_patients), min(10, len(all_patients)))
    
    # 逐层扩散
    layer1 = df[df['head'].isin(selected_patients)]
    intermediate_nodes = layer1['tail'].unique()
    layer2 = df[df['head'].isin(intermediate_nodes)]
    # 仅保留 PrimeKG 内部连接
    prime_nodes = layer2[layer2['tail'].astype(str).str.contains("PrimeKG", na=False)]['tail'].unique()
    layer3 = df[(df['head'].isin(prime_nodes)) & (df['tail'].isin(prime_nodes))]
    
    subset_df = pd.concat([layer1, layer2, layer3]).drop_duplicates()
else:
    subset_df = df.head(100) # 兜底

print(f"📊 绘图数据准备完毕: {len(subset_df)} 条边")

# --- 🕸️ 构建 NetworkX 图 ---
G = nx.Graph() # 使用无向图布局会更好看，如需有向改 nx.DiGraph()

# 收集所有节点并确定颜色/大小
node_colors = []
node_sizes = []
node_list = []

# 先添加边
for _, row in subset_df.iterrows():
    G.add_edge(str(row['head']), str(row['tail']))

# 计算布局 (Spring 布局最适合知识图谱)
# k参数调节节点间距，越大越松散
print("⏳ 正在计算物理布局 (可能需要几秒)...")
pos = nx.spring_layout(G, k=0.15, iterations=50, seed=42)

# 为每个节点分配样式
for node in G.nodes():
    cat = get_node_category(node)
    color = COLORS_MAP.get(cat, COLORS_MAP["Other"])
    
    # 大小逻辑
    size = 30
    if cat == "Patient": size = 300
    elif cat == "Gene": size = 250
    elif cat == "PrimeKG_Disease": size = 150
    elif cat == "PrimeKG_Gene": size = 100
    elif cat == "Symptom": size = 80
    
    node_colors.append(color)
    node_sizes.append(size)
    node_list.append(node)

# ================= 3. Matplotlib 绘图 =================
print(f"🎨 开始渲染高分辨率图片 ({DPI} DPI)...")

# 创建画布，figsize 控制长宽比例 (单位英寸)
plt.figure(figsize=(20, 20), dpi=DPI) 

# 1. 画节点
nx.draw_networkx_nodes(G, pos, 
                       nodelist=node_list, 
                       node_color=node_colors, 
                       node_size=node_sizes, 
                       alpha=0.9, 
                       edgecolors='white', # 节点白边
                       linewidths=0.5)

# 2. 画边
nx.draw_networkx_edges(G, pos, 
                       width=0.3, 
                       alpha=0.3, 
                       edge_color='gray')

# 3. 画标签 (仅给重要节点打标签，避免重叠)
# 策略：Patient 和 Gene 必打标签，其他随机或不打
labels = {}
for node in G.nodes():
    cat = get_node_category(node)
    if cat in ["Patient", "Gene", "Symptom"]: 
        labels[node] = node
    elif cat == "PrimeKG_Disease" and random.random() > 0.8: # 抽样显示
        labels[node] = node

nx.draw_networkx_labels(G, pos, labels, font_size=4, font_color='black')

# 4. 手动制作图例 (Legend)
legend_patches = [mpatches.Patch(color=color, label=cat) for cat, color in COLORS_MAP.items()]
plt.legend(handles=legend_patches, loc='upper right', prop={'size': 12}, title="Node Types")

plt.title("ADNI Knowledge Graph Subset (High-Res)", fontsize=20)
plt.axis('off') # 关闭坐标轴

# ================= 4. 保存 =================
plt.savefig(OUTPUT_PNG, format='png', bbox_inches='tight')
# 如果想要 PDF，取消下面这行的注释
# plt.savefig(OUTPUT_PNG.replace(".png", ".pdf"), format='pdf', bbox_inches='tight')

print(f"✅ 图片已保存: {OUTPUT_PNG}")
plt.close()

🚀 正在加载数据: adni_knowledge_triplets.csv ...
🔍 正在抽取 10 人子图数据...
📊 绘图数据准备完毕: 263 条边
⏳ 正在计算物理布局 (可能需要几秒)...
🎨 开始渲染高分辨率图片 (600 DPI)...
✅ 图片已保存: adni_kg_subset_high_res.png
